# Stage 4 G2 — Tiny RL sanity on Qwen3.6-35B-A3B + L23 SAE, SuperGPQA

**Goal**: replicate the three-way reward ablation from Qwen3.5-4B G2 on the new architecture.

- **R0** — outcome only (SuperGPQA letter exact match)
- **R1** — outcome + **SAE-sparse** per-token reward (our helpful−harmful pack, TopK encoded)
- **R2** — outcome + **raw direction** per-token reward (projection onto helpful−harmful vector in W_dec residual space, no sparsity)

**What we're looking for**: R1 ≥ R0 + 2 pp on held-out eval, AND R2 ≤ R0 (replicate the 11 pp R1-vs-R2 gap from Qwen3.5-4B G2). If both hold, this is the **central paper ablation** confirming that sparse decomposition beats raw-direction for reasoning on hybrid-architecture MoE.

**Inputs**:
- Qwen3.6-35B-A3B base (~70 GB bf16)
- SAE at `caiovicentino1/Qwen3.6-35B-A3B-SAE-L23-topk-wip/sae_step_22674_92M_tokens.pt`
- Feature pack at `catalogs/qwen3.6-35b-a3b/reasoning_pack.json` (ρ=0.5217 validated at G1)

**Scale** (reduced from Qwen3.5-4B G2 because 35B × thinking-mode × long responses costs 80× more per step):
- 30 steps per ablation (vs 100)
- 2 questions × 2 rollouts = 4 rollouts/step (vs 16)
- max_gen_len=1536 (thinking mode can truncate; accept slight accuracy noise)
- 3 ablations sequential

**Budget**: ~6 h on RTX 6000 Blackwell 96 GB, ~$20. Disjoint from G1 validation set.

**Pre-registered thresholds**:
- **PASS C2**: R1 final accuracy ≥ R0 + 2 pp on 100-question held-out eval.
- **PASS sparse>raw**: R1 − R2 ≥ 5 pp (smaller gap than Qwen3.5's 11 pp still counts; noise budget larger with 30 steps).
- **FAIL**: R1 ≈ R0 → features are not informative enough as dense reward. Diagnose before going G3.

## Cell 1 — Install env (fla for GDN fast path; transformers main for qwen3_5_moe)

In [ ]:
import sys, subprocess, os, shutil, site
from pathlib import Path

def _pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check)

def _check():
    try:
        import transformers
        try:
            from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
        except ImportError:
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
        return 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, transformers.__version__
    except Exception as e:
        return False, f'import-error: {e}'

ok, ver = _check()
print(f'Current: transformers={ver}, qwen3_5_moe={ok}')

if not ok:
    print('\n→ Installing transformers @ main + peer deps')
    _pip('install', '-q', 'accelerate', 'peft', 'trl', 'datasets',
         'huggingface_hub==1.5.0', 'safetensors', 'sympy', 'einops',
         'scikit-learn', 'matplotlib', 'sentencepiece', 'tokenizers',
         'protobuf', 'scipy')
    _pip('uninstall', '-y', 'transformers', check=False)
    _pip('uninstall', '-y', 'transformers', check=False)
    for p in sys.path + site.getsitepackages() + [site.getusersitepackages()]:
        tr = Path(p) / 'transformers'
        if tr.exists():
            shutil.rmtree(tr, ignore_errors=True)
    SRC_DIR = '/content/transformers_src'
    if os.path.exists(SRC_DIR):
        shutil.rmtree(SRC_DIR)
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/huggingface/transformers.git', SRC_DIR], check=True)
    _pip('install', '--force-reinstall', '--no-deps', '--no-cache-dir', SRC_DIR)
    # fla pra speedup no GDN (Python puro, confiavel)
    _pip('install', '--no-cache-dir', 'flash-linear-attention', check=False)
    # causal-conv1d opcional, Qwen3.6 GDN nao usa tanto mas vale tentar
    _pip('install', '--no-build-isolation', '--no-cache-dir', 'causal-conv1d', check=False)

    for mod in list(sys.modules):
        if mod.startswith('transformers') or mod.startswith('huggingface_hub'):
            del sys.modules[mod]
    ok, ver = _check()
    print(f'Post-install: transformers={ver}, qwen3_5_moe={ok}')
    if not ok:
        print('\n⚠️  Restart kernel and re-run.')
        raise SystemExit

# HF auth + Drive
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('✅ HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

from google.colab import drive
drive.mount('/content/drive')

# Global imports
import json, time, random, re, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from contextlib import nullcontext

DRIVE_ROOT = Path('/content/drive/MyDrive/mechreward/s4_qwen36')
G2_ROOT = DRIVE_ROOT / 'g2'
G2_ROOT.mkdir(parents=True, exist_ok=True)
print(f'\nG2 artifacts: {G2_ROOT}')

## Cell 2 — CFG

In [ ]:
CFG = dict(
    # Model + SAE
    model_id='Qwen/Qwen3.6-35B-A3B',
    sae_repo='caiovicentino1/Qwen3.6-35B-A3B-SAE-L23-topk-wip',
    sae_file='sae_step_22674_92M_tokens.pt',
    sae_layer=23,
    pack_repo='caiovicentino/mechreward',
    pack_path='catalogs/qwen3.6-35b-a3b/reasoning_pack.json',

    # Training (compact for 35B MoE + thinking mode)
    max_steps=30,
    batch_questions=2,
    rollouts_per_q=2,
    micro_batch_logprobs=1,
    max_prompt_len=512,
    max_gen_len=1536,
    lr=1e-6,                   # G2 recipe from Qwen3.5; bump if KL stuck like G3
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    lora_target=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],

    # Rewards
    lam=0.1,                   # mech weight vs outcome — validated on Qwen3.5 G2
    kl_beta=0.05,
    per_token_reward=True,

    # Eval
    quick_eval_every=10,
    quick_eval_n=30,
    final_eval_n=100,

    # Data (disjoint from G1 which used [100:250])
    difficulty_filter=['easy', 'middle'],
    train_split='[350:550]',
    eval_split='[550:650]',
    seed=42,

    # Output
    ckpt_dir=str(G2_ROOT),
)
print(json.dumps({k: v for k, v in CFG.items() if not k.startswith('_')}, indent=2))

## Cell 3 — Load Qwen3.6-35B-A3B + tokenizer

In [ ]:
import transformers
try:
    from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
except ImportError:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
assert 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, 'Restart kernel, re-run Cell 1.'

from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, PeftModel

tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForImageTextToText.from_pretrained(
    CFG['model_id'],
    dtype=torch.bfloat16,
    device_map='cuda',
    attn_implementation='sdpa',
    trust_remote_code=True,
)

# Freeze vision tower
for n, p in base_model.named_parameters():
    if 'visual' in n.lower() or 'vision' in n.lower():
        p.requires_grad = False

print(f'Base model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 4 — Load SAE + feature pack

In [ ]:
from huggingface_hub import hf_hub_download

# Download SAE snapshot
sae_ckpt_path = hf_hub_download(repo_id=CFG['sae_repo'], filename=CFG['sae_file'])
print(f'SAE ckpt: {sae_ckpt_path}')

sae_state = torch.load(sae_ckpt_path, map_location='cuda', weights_only=False)
if hasattr(sae_state, 'state_dict'):
    sae_state = sae_state.state_dict()

W_enc = sae_state['W_enc'].to('cuda', torch.float32)
W_dec = sae_state['W_dec'].to('cuda', torch.float32)
if W_enc.shape[0] > W_enc.shape[1]:
    W_enc = W_enc.t().contiguous()
if W_dec.shape[0] < W_dec.shape[1]:
    W_dec = W_dec.t().contiguous()
d_model, d_sae = W_enc.shape
b_enc = sae_state.get('b_enc', torch.zeros(d_sae, device='cuda')).to('cuda', torch.float32)
b_dec = sae_state.get('b_dec', torch.zeros(d_model, device='cuda')).to('cuda', torch.float32)
K = int(sae_state.get('k', 128))
print(f'SAE: d_model={d_model}, d_sae={d_sae}, k={K}')

# Feature pack from github
if not Path('/content/mechreward').exists():
    subprocess.run(['git', 'clone', '-q', f'https://github.com/{CFG["pack_repo"]}.git',
                    '/content/mechreward'], check=True)
with open(f'/content/mechreward/{CFG["pack_path"]}') as f:
    pack = json.load(f)

helpful_ids = [x['feature_id'] for x in pack['helpful_features']]
harmful_ids = [x['feature_id'] for x in pack['harmful_features']]
ALL_FEATS = torch.tensor(helpful_ids + harmful_ids, dtype=torch.long, device='cuda')
FEAT_WEIGHTS = torch.tensor([1.0]*len(helpful_ids) + [-1.0]*len(harmful_ids),
                            dtype=torch.float32, device='cuda')
# Selected columns of W_enc for faster per-token encode in R1
W_enc_sel = W_enc[:, ALL_FEATS].contiguous()
b_enc_sel = b_enc[ALL_FEATS].contiguous()
print(f'Pack loaded: {len(helpful_ids)} helpful + {len(harmful_ids)} harmful (ρ=0.5217 from G1)')

# R2 direction: mean decoder vector of helpful - mean decoder vector of harmful, normalized
# This is the "raw direction" at L23 implied by the same contrastive features but WITHOUT sparsity.
with torch.no_grad():
    W_dec_helpful_mean = W_dec[torch.tensor(helpful_ids, device='cuda')].mean(dim=0)
    W_dec_harmful_mean = W_dec[torch.tensor(harmful_ids, device='cuda')].mean(dim=0)
    raw_direction = W_dec_helpful_mean - W_dec_harmful_mean
    raw_direction = raw_direction / (raw_direction.norm() + 1e-8)
print(f'Raw direction (R2) shape: {tuple(raw_direction.shape)}, norm: {raw_direction.norm().item():.3f} (should be 1.0)')

## Cell 5 — Per-token rewards (R1 sparse, R2 raw) + hook

In [ ]:
def get_layer_module(m, idx):
    candidates = [m]
    if hasattr(m, 'base_model') and m.base_model is not m:
        candidates.append(m.base_model.model if hasattr(m.base_model, 'model') else m.base_model)
    if hasattr(m, 'model'):
        candidates.append(m.model)
    paths = [
        ('model', 'language_model', 'layers'),
        ('language_model', 'layers'),
        ('model', 'layers'),
        ('layers',),
    ]
    for start in candidates:
        for path in paths:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except (IndexError, TypeError): continue
    raise RuntimeError(f'Could not locate layer {idx}')

class HiddenCapture:
    def __init__(self): self.h = None
    def hook(self, module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        self.h = h.detach()

def register_capture(m, layer_idx):
    cap = HiddenCapture()
    handle = get_layer_module(m, layer_idx).register_forward_hook(cap.hook)
    return cap, handle

def mech_reward_sparse_per_token(hidden, attn_mask):
    """R1 reward: TopK-selected helpful/harmful feature activations, summed with signs.
    Uses selective encode (only on target features) for speed.
    hidden: [B, T, d_model] (bf16 or fp32). Returns [B, T] fp32.
    
    NOTE: this is a *selective* approximation — we encode only the 20 target features,
    not the full d_sae=32768. For the target features, the pre-activation is identical
    to a full TopK encode (because TopK operates per-token across all features, but
    selecting before TopK is wrong if the target feature is NOT in the top-K).
    
    For the helpful/harmful pack, we validated in G1 that this selective encode matches
    within 99% of the full TopK encode on typical tokens (the pack features are high-
    activation by construction). We accept the 1% approximation for 20x speedup.
    """
    h = hidden.to(torch.float32) - b_dec
    pre = torch.einsum('btd,df->btf', h, W_enc_sel) + b_enc_sel  # [B, T, 20]
    acts = F.relu(pre)
    r = (acts * FEAT_WEIGHTS).sum(dim=-1) * attn_mask.to(acts.dtype)
    return r

def mech_reward_raw_per_token(hidden, attn_mask):
    """R2 reward: raw projection of residual onto helpful-minus-harmful direction in W_dec space.
    No sparsity, no ReLU. Continuous signed signal.
    hidden: [B, T, d_model]. Returns [B, T] fp32.
    """
    h = hidden.to(torch.float32) - b_dec  # center same as SAE does
    r = torch.einsum('btd,d->bt', h, raw_direction) * attn_mask.to(torch.float32)
    return r

def mech_reward_zero_per_token(hidden, attn_mask):
    """R0: no mech reward (only outcome is used)."""
    return torch.zeros_like(attn_mask, dtype=torch.float32)

REWARD_FN = {
    'R0': mech_reward_zero_per_token,
    'R1': mech_reward_sparse_per_token,
    'R2': mech_reward_raw_per_token,
}
print('Reward functions registered:', list(REWARD_FN.keys()))

# Sanity: run a test input through both R1 and R2 — should produce different signatures
with torch.inference_mode():
    probe_h = torch.randn(1, 10, d_model, device='cuda')
    probe_mask = torch.ones(1, 10, device='cuda')
    r_sparse = mech_reward_sparse_per_token(probe_h, probe_mask)
    r_raw = mech_reward_raw_per_token(probe_h, probe_mask)
print(f'R1 sparse on random: range [{r_sparse.min():.3f}, {r_sparse.max():.3f}]')
print(f'R2 raw on random:    range [{r_raw.min():.3f}, {r_raw.max():.3f}]')

## Cell 6 — Dataset + prompt + verifier (SuperGPQA, disjoint from G1)

In [ ]:
from datasets import load_dataset

raw = load_dataset('m-a-p/SuperGPQA', split='train')
filtered = raw.filter(lambda ex: ex['difficulty'] in CFG['difficulty_filter'])
shuffled = filtered.shuffle(seed=CFG['seed'])

# Disjoint splits from S4-L0 [0:100] + G1 [100:250]
# Train: [350:550] (200 questions); Eval: [550:650] (100 questions)
SGPQA_TRAIN = shuffled.select(range(350, 550))
SGPQA_EVAL_FULL = shuffled.select(range(550, 650))
# Quick eval: first 30 of full eval
SGPQA_EVAL_QUICK = SGPQA_EVAL_FULL.select(range(CFG['quick_eval_n']))
print(f'Train: {len(SGPQA_TRAIN)}, Eval full: {len(SGPQA_EVAL_FULL)}, Eval quick: {len(SGPQA_EVAL_QUICK)}')

def format_prompt(ex):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(ex['options']))
    msgs = [{
        'role': 'user',
        'content': (
            f"Answer the following multiple-choice question. Reason step by step, "
            f"then put the letter of the correct answer in \\boxed{{}}.\n\n"
            f"Question: {ex['question']}\n\n"
            f"Options:\n{choices}"
        )
    }]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
LETTER_AT_END_RE = re.compile(r'[Aa]nswer[:\s]+([A-J])\b')

def extract_letter(completion, n_opts):
    m = list(BOXED_RE.finditer(completion))
    if m:
        letter = m[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts:
            return letter
    m2 = list(LETTER_AT_END_RE.finditer(completion))
    if m2:
        letter = m2[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts:
            return letter
    tail = completion[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands:
        letter = cands[-1]
        if ord(letter) - ord('A') < n_opts:
            return letter
    return None

def verify(ex, completion):
    pred = extract_letter(completion, len(ex['options']))
    return pred is not None and pred == ex['answer_letter']

## Cell 7 — Eval harness

In [ ]:
@torch.no_grad()
def generate_batch(model, prompts, max_new=None, temperature=0.0, n=1):
    max_new = max_new or CFG['max_gen_len']
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    outs = []
    for p in prompts:
        enc = tok(p, return_tensors='pt', truncation=True,
                  max_length=CFG['max_prompt_len']).to('cuda')
        gen = model.generate(
            **enc, max_new_tokens=max_new,
            do_sample=temperature > 0, temperature=max(temperature, 1e-5),
            top_p=1.0, pad_token_id=tok.pad_token_id,
            num_return_sequences=n, use_cache=True,
        )
        for g in gen:
            outs.append(tok.decode(g[enc.input_ids.shape[1]:], skip_special_tokens=True))
    return outs

def eval_sgpqa(model, dataset, max_new=None):
    correct = 0
    for ex in dataset:
        prompt = format_prompt(ex)
        out = generate_batch(model, [prompt], max_new=max_new, temperature=0.0)[0]
        if verify(ex, out):
            correct += 1
    return correct / len(dataset)

def run_quick_eval(model):
    model.eval()
    # Use shorter max_new for quick eval to save time
    acc = eval_sgpqa(model, SGPQA_EVAL_QUICK, max_new=min(1024, CFG['max_gen_len']))
    model.train()
    return {'quick_eval': acc}

def run_full_eval(model):
    model.eval()
    acc = eval_sgpqa(model, SGPQA_EVAL_FULL, max_new=CFG['max_gen_len'])
    model.train()
    return {'final_eval': acc}

print('Eval harness ready')

## Cell 8 — GRPO step (takes `mode` in {R0, R1, R2})

In [ ]:
def compute_logprobs(m, input_ids, attn_mask, response_mask, micro=None):
    """Chunked over batch, bf16 log_softmax."""
    micro = micro or CFG['micro_batch_logprobs']
    B = input_ids.shape[0]
    outs = []
    for i in range(0, B, micro):
        ids_c = input_ids[i:i+micro]
        attn_c = attn_mask[i:i+micro]
        resp_c = response_mask[i:i+micro]
        out = m(input_ids=ids_c, attention_mask=attn_c, use_cache=False)
        logits = out.logits[:, :-1]
        targets = ids_c[:, 1:]
        logp = F.log_softmax(logits, dim=-1)
        tok_logp = logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1).float()
        outs.append(tok_logp * resp_c[:, 1:].float())
        del out, logits, logp, tok_logp
    return torch.cat(outs, dim=0)

def grpo_step(model, batch, step, mode):
    """mode in {R0, R1, R2}. Returns (loss, log_dict)."""
    reward_fn = REWARD_FN[mode]
    lam = 0.0 if mode == 'R0' else CFG['lam']
    kl_c = CFG['kl_beta']

    prompts = [format_prompt(ex) for ex in batch]
    golds = [ex['answer_letter'] for ex in batch]
    n_opts_list = [len(ex['options']) for ex in batch]

    all_ids, all_attn, all_resp, all_outcome, all_mech_tok = [], [], [], [], []

    # ROLLOUT
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    cap, handle = register_capture(model, CFG['sae_layer'])
    try:
        for pi, prompt in enumerate(prompts):
            enc = tok(prompt, return_tensors='pt', truncation=True,
                      max_length=CFG['max_prompt_len']).to('cuda')
            P = enc.input_ids.shape[1]
            with torch.no_grad():
                gen = model.generate(
                    **enc, max_new_tokens=CFG['max_gen_len'],
                    do_sample=True, temperature=0.9, top_p=0.95,
                    num_return_sequences=CFG['rollouts_per_q'],
                    pad_token_id=tok.pad_token_id, use_cache=True,
                )
            attn = (gen != tok.pad_token_id).long().to('cuda')
            # Second forward to capture hidden at L23
            with torch.no_grad():
                model(input_ids=gen, attention_mask=attn, use_cache=False)
            hidden = cap.h
            resp_mask = torch.zeros_like(attn)
            resp_mask[:, P:] = attn[:, P:]
            # Reward per token (zero for R0)
            mech_tok = reward_fn(hidden, resp_mask)
            # Outcome per rollout
            completions = tok.batch_decode(gen[:, P:], skip_special_tokens=True)
            outcomes = torch.tensor([
                1.0 if verify(batch[pi], c) else 0.0 for c in completions
            ], device='cuda')

            all_ids.append(gen); all_attn.append(attn); all_resp.append(resp_mask)
            all_outcome.append(outcomes); all_mech_tok.append(mech_tok.detach())
            del hidden
    finally:
        handle.remove()
    torch.cuda.empty_cache()

    # Pad + concat
    maxT = max(x.shape[1] for x in all_ids)
    def pad_right(x, val=0):
        pad = maxT - x.shape[1]
        return F.pad(x, (0, pad), value=val) if pad > 0 else x
    ids = torch.cat([pad_right(x, tok.pad_token_id) for x in all_ids], dim=0)
    attn = torch.cat([pad_right(x) for x in all_attn], dim=0)
    resp = torch.cat([pad_right(x) for x in all_resp], dim=0)
    mech_tok = torch.cat([pad_right(x) for x in all_mech_tok], dim=0)
    outcomes = torch.cat(all_outcome, dim=0)

    lens = resp.sum(dim=-1).clamp(min=1).float()
    mech_traj = mech_tok.sum(dim=-1) / lens
    traj_r = outcomes + lam * mech_traj
    N = CFG['rollouts_per_q']
    traj_r = traj_r.view(-1, N)
    adv = (traj_r - traj_r.mean(dim=-1, keepdim=True)) / (traj_r.std(dim=-1, keepdim=True) + 1e-8)
    adv = adv.view(-1)
    adv_tok = adv.unsqueeze(-1) * resp[:, 1:].float()
    if CFG['per_token_reward'] and mode != 'R0':
        mt = mech_tok[:, 1:].view(-1, N, maxT - 1)
        mt = (mt - mt.mean(dim=(1,2), keepdim=True)) / (mt.std(dim=(1,2), keepdim=True) + 1e-8)
        adv_tok = adv_tok + lam * mt.view(-1, maxT - 1) * resp[:, 1:].float()

    # TRAIN
    model.train()
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    try: model.enable_input_require_grads()
    except Exception: pass

    pol_logp = compute_logprobs(model, ids, attn, resp)
    with torch.no_grad(), model.disable_adapter():
        ref_logp = compute_logprobs(model, ids, attn, resp)
    kl = pol_logp - ref_logp
    denom = resp[:, 1:].sum().clamp(min=1).float()
    loss = -(pol_logp * adv_tok).sum() / denom
    loss = loss + kl_c * (kl ** 2).sum() / denom

    return loss, dict(
        mode=mode, lam=lam,
        mean_outcome=outcomes.mean().item(),
        mean_mech=mech_traj.mean().item(),
        mean_kl=(kl.sum() / denom).item(),
    )

print('grpo_step ready')

## Cell 9 — Run one ablation end-to-end

In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm

def run_ablation(mode):
    """Train with the given reward mode, return history + final eval."""
    print(f'\n{"="*70}\n=== Running ablation: {mode} ===\n{"="*70}\n')

    ablation_dir = G2_ROOT / mode
    ablation_dir.mkdir(parents=True, exist_ok=True)

    # Fresh LoRA on top of base
    lora_cfg = LoraConfig(
        r=CFG['lora_r'],
        lora_alpha=CFG['lora_alpha'],
        lora_dropout=CFG['lora_dropout'],
        target_modules=CFG['lora_target'],
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()
    optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG['lr'])

    # Pre-train eval
    t0 = time.time()
    pre = run_quick_eval(model)
    print(f'[{mode}] pre-train quick eval: {pre["quick_eval"]*100:.1f}%')

    history = [{'step': 0, 'mode': mode, **pre, 'elapsed': time.time()-t0}]

    # Shuffle train
    random.seed(CFG['seed'])
    indices = list(range(len(SGPQA_TRAIN)))
    random.shuffle(indices)

    for step in range(CFG['max_steps']):
        off = (step * CFG['batch_questions']) % len(SGPQA_TRAIN)
        batch_idx = [indices[(off + i) % len(SGPQA_TRAIN)] for i in range(CFG['batch_questions'])]
        batch = [SGPQA_TRAIN[i] for i in batch_idx]

        t_step = time.time()
        loss, log = grpo_step(model, batch, step + 1, mode)
        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optim.step()
        dt = time.time() - t_step

        log.update(step=step + 1, loss=loss.item(), step_sec=dt, elapsed=time.time()-t0)

        if (step + 1) % CFG['quick_eval_every'] == 0:
            qe = run_quick_eval(model)
            log.update(qe)
            # Save adapter snapshot
            model.save_pretrained(str(ablation_dir / f'step_{step+1}'))

        history.append(log)
        if (step + 1) % 2 == 0 or step == 0:
            print(f"[{mode} step {step+1:2d}] loss={log['loss']:.4f} "
                  f"outcome={log['mean_outcome']:.2f} mech={log['mean_mech']:.3f} "
                  f"kl={log['mean_kl']:.4f} {dt:.0f}s"
                  + (f" quick={log.get('quick_eval', 0)*100:.0f}%" if 'quick_eval' in log else ''))

    # Final full eval
    print(f'\n[{mode}] running final eval ({CFG["final_eval_n"]}Q)...')
    final = run_full_eval(model)
    print(f'[{mode}] final accuracy: {final["final_eval"]*100:.2f}% '
          f'(vs pre-train {pre["quick_eval"]*100:.1f}% on 30Q)')

    # Save final adapter + history
    model.save_pretrained(str(ablation_dir / 'final'))
    with open(ablation_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2, default=float)
    with open(ablation_dir / 'final.json', 'w') as f:
        json.dump({'mode': mode, 'pre_train_eval30': pre['quick_eval'],
                   'final_eval100': final['final_eval'],
                   'elapsed_min': (time.time()-t0)/60}, f, indent=2)

    # Unload LoRA to free memory for next ablation
    del model, optim
    gc.collect(); torch.cuda.empty_cache()
    print(f'VRAM after unload: {torch.cuda.memory_allocated()/1e9:.1f} GB')

    return {'mode': mode, 'pre': pre['quick_eval'], 'final': final['final_eval'],
            'history': history, 'elapsed_min': (time.time()-t0)/60}

## Cell 10 — Run three ablations sequentially

In [ ]:
results = {}
total_start = time.time()
for mode in ['R0', 'R1', 'R2']:
    results[mode] = run_ablation(mode)

elapsed_total_h = (time.time() - total_start) / 3600
print(f'\n{"="*70}')
print(f'ALL 3 ABLATIONS DONE ({elapsed_total_h:.1f} h total)')
print(f'{"="*70}')
for mode in ['R0', 'R1', 'R2']:
    r = results[mode]
    print(f'{mode}: pre={r["pre"]*100:.1f}% final={r["final"]*100:.2f}% '
          f'Δ {(r["final"]-r["pre"])*100:+.1f}pp ({r["elapsed_min"]:.0f}min)')

## Cell 11 — Summary + verdict

In [ ]:
r0 = results['R0']['final']
r1 = results['R1']['final']
r2 = results['R2']['final']

r1_vs_r0_pp = (r1 - r0) * 100
r1_vs_r2_pp = (r1 - r2) * 100
r2_vs_r0_pp = (r2 - r0) * 100

print(f'{"="*70}')
print(f'STAGE GATE 2 — VERDICT')
print(f'{"="*70}')
print(f'R0 (outcome only):                 {r0*100:.2f}%')
print(f'R1 (outcome + SAE-sparse, λ=0.1):  {r1*100:.2f}%  (Δ vs R0: {r1_vs_r0_pp:+.1f}pp)')
print(f'R2 (outcome + raw direction, λ=0.1): {r2*100:.2f}%  (Δ vs R0: {r2_vs_r0_pp:+.1f}pp)')
print()
print(f'R1 - R2 gap (sparse vs raw):       {r1_vs_r2_pp:+.1f}pp')
print(f'Qwen3.5-4B G2 reference:           R1-R2 = +11pp')
print()

# Pre-registered thresholds
pass_c2 = r1_vs_r0_pp >= 2.0
pass_sparse_beats_raw = r1_vs_r2_pp >= 5.0

print(f'PASS C2 (R1 ≥ R0 + 2pp):    {"✅ YES" if pass_c2 else "❌ NO"}  ({r1_vs_r0_pp:+.1f}pp)')
print(f'PASS sparse > raw (+5pp):  {"✅ YES" if pass_sparse_beats_raw else "❌ NO"}  ({r1_vs_r2_pp:+.1f}pp)')

if pass_c2 and pass_sparse_beats_raw:
    verdict = 'PASS-EXCELLENT: G2 replicates Qwen3.5-4B pattern. Go S4-G3.'
elif pass_c2:
    verdict = 'PASS-PARTIAL: C2 met but sparse-vs-raw gap below 5pp. Go G3, investigate R2 separately.'
elif r1_vs_r0_pp > 0 and r2_vs_r0_pp < 0:
    verdict = 'MARGINAL: signs correct (R1>0, R2<0) but magnitudes small. Could be noise; consider rerun with more steps.'
else:
    verdict = 'FAIL: investigate. Possible causes: feature pack insufficient, layer wrong, LR too low.'

print(f'\nVerdict: {verdict}')

# Save master summary
summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model': CFG['model_id'],
    'sae_ckpt': CFG['sae_file'],
    'sae_layer': CFG['sae_layer'],
    'config': {k: v for k, v in CFG.items() if k not in ('ckpt_dir',)},
    'results': {
        'R0_final': r0, 'R1_final': r1, 'R2_final': r2,
        'R1_vs_R0_pp': r1_vs_r0_pp,
        'R1_vs_R2_pp': r1_vs_r2_pp,
        'R2_vs_R0_pp': r2_vs_r0_pp,
    },
    'verdict': verdict,
    'pass_c2': pass_c2,
    'pass_sparse_beats_raw': pass_sparse_beats_raw,
    'elapsed_h': (time.time() - total_start) / 3600,
}
with open(G2_ROOT / 's4_g2_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'\n✅ Saved: {G2_ROOT}/s4_g2_summary.json')

## Cell 12 — Upload summary to HF for redundancy (optional)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
try:
    api.upload_file(
        path_or_fileobj=str(G2_ROOT / 's4_g2_summary.json'),
        path_in_repo='s4_g2_summary.json',
        repo_id='caiovicentino1/Qwen3.6-35B-A3B-SAE-L23-topk-wip',
        repo_type='model',
    )
    print('✅ Uploaded summary to HF')
except Exception as e:
    print(f'HF upload failed (not critical): {e}')